In [1]:
from sem2_old import sem2

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch import distributions as dist
from nn_models import MLP  
import numpy as np
from engression.loss_func import energy_loss, energy_loss_two_sample

In [7]:
def generate_true_data(n_samples, beta=None):
    u1 = dist.Exponential(1.0).sample((n_samples,1)) 
    u2 = dist.Uniform(low=0.0, high=1.0).sample((n_samples,1))
    u3=dist.Normal(loc=0.0, scale=1.0).sample((n_samples,1))
    x = torch.sin(u1) +4*u2**3-u2**2  
    y = x*beta[0] + u3*beta[1]+1 
    return torch.cat([u1, u2,u3, x, y], dim=1)

model_formula = 'u1 ~ Exponential(1.0); u2 ~ Uniform(0,1);u3 ~ Normal(0,1); x <- n(u1,u2); y <- x +u3'
model_components={
    "x": MLP(input_dim=2, hidden_dims=[64,64], output_dim=1)
}
model = sem2(model_formula=model_formula, model_components=model_components)

beta_true=[2,1]
dataset=generate_true_data(3000,beta=beta_true)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
model.train()
niter = 5000
for i in range(niter):
    optimizer.zero_grad()
    sample = model(dataset.size(0) * 2)
    loss, s1, s2 = energy_loss(dataset, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 1000 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

model.eval()
with torch.no_grad():
        beta_hat = model.layer['y'].weight.detach().cpu().numpy().flatten()
        bias_hat = model.layer['y'].bias.detach().cpu().numpy().flatten()
print(f"\nTrue beta: {beta_true}")
print(f"Estimated beta: {beta_hat.tolist()}")
print(f"True intercept: {[1]}；Estimated intercept: {bias_hat.tolist()}")
print(f"Beta estimation error: {np.abs(beta_true - beta_hat).tolist()}")

def predict_x(u1, u2):
    with torch.no_grad():
        inputs = torch.cat([u1, u2], dim=1)
        x_pred = model.layer['x'](inputs)
    return x_pred
    
def predict_y(u1, u2, u3):
    with torch.no_grad():
        x_pred = predict_x(u1, u2)
        inputs = torch.cat([x_pred, u3], dim=1)
        y_pred = model.layer['y'](inputs)
    return y_pred

test_data = generate_true_data(n_samples=1000, beta=beta_true)
u1_test, u2_test, u3_test, x_true, y_true = (
    test_data[:, 0:1],
    test_data[:, 1:2],
    test_data[:, 2:3],
    test_data[:, 3:4],
    test_data[:, 4:5]
)

with torch.no_grad():
    x_pred = predict_x(u1_test, u2_test)
    y_pred = predict_y(u1_test, u2_test, u3_test)

mse_x = torch.mean((x_pred - x_true) ** 2).item()
mse_y = torch.mean((y_pred - y_true) ** 2).item()

print(f"\nMSE for x (on 1000 test samples): {mse_x:.4f}")
print(f"MSE for y (on 1000 test samples): {mse_y:.4f}")

Iter 1 | loss: 3.2029, s1: 4.1883, s2: 1.9708
Iter 1000 | loss: 1.7226, s1: 3.4143, s2: 3.3833
Iter 2000 | loss: 1.6653, s1: 3.3635, s2: 3.3963
Iter 3000 | loss: 1.6915, s1: 3.3394, s2: 3.2960
Iter 4000 | loss: 1.6836, s1: 3.3947, s2: 3.4224
Iter 5000 | loss: 1.6709, s1: 3.3559, s2: 3.3700

True beta: [2, 1]
Estimated beta: [1.7552083730697632, 1.034342885017395]
True intercept: [1]；Estimated intercept: [1.2965892553329468]
Beta estimation error: [0.24479162693023682, 0.03434288501739502]

MSE for x (on 1000 test samples): 0.0598
MSE for y (on 1000 test samples): 0.1689


In [15]:
def generate_true_data(n_samples, beta=None):
    u1 = dist.Exponential(1.0).sample((n_samples,1)) 
    u2 = dist.Uniform(low=0.0, high=1.0).sample((n_samples,1))
    u3=dist.Normal(loc=0.0, scale=1.0).sample((n_samples,1))
    x = torch.sin(u1) +4*u2**3-u2**2  
    y = x*beta[0] + u3*beta[1]+1 
    return torch.cat([u1, u2,u3, x, y], dim=1)

model_formula = 'u1 ~ Exponential(1.0); u2 ~ Uniform(0,1);u3 ~ Normal(0,1); x <- n(u1,u2); y <- x +u3'
model_components={
    "x": MLP(input_dim=2, hidden_dims=[64,64], output_dim=1)
}
model = sem2(model_formula=model_formula, model_components=model_components)

beta_true=[2,1]
dataset=generate_true_data(3000,beta=beta_true)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
model.train()
niter = 8000
for i in range(niter):
    optimizer.zero_grad()
    sample = model(dataset.size(0) * 2)
    loss, s1, s2 = energy_loss(dataset, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 1000 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

model.eval()
with torch.no_grad():
        beta_hat = model.layer['y'].weight.detach().cpu().numpy().flatten()
        bias_hat = model.layer['y'].bias.detach().cpu().numpy().flatten()
print(f"\nTrue beta: {beta_true}")
print(f"Estimated beta: {beta_hat.tolist()}")
print(f"True intercept: {[1]}；Estimated intercept: {bias_hat.tolist()}")
print(f"Beta estimation error: {np.abs(beta_true - beta_hat).tolist()}")

def predict_x(u1, u2):
    with torch.no_grad():
        inputs = torch.cat([u1, u2], dim=1)
        x_pred = model.layer['x'](inputs)
    return x_pred
    
def predict_y(u1, u2, u3):
    with torch.no_grad():
        x_pred = predict_x(u1, u2)
        inputs = torch.cat([x_pred, u3], dim=1)
        y_pred = model.layer['y'](inputs)
    return y_pred

test_data = generate_true_data(n_samples=1000, beta=beta_true)
u1_test, u2_test, u3_test, x_true, y_true = (
    test_data[:, 0:1],
    test_data[:, 1:2],
    test_data[:, 2:3],
    test_data[:, 3:4],
    test_data[:, 4:5]
)

with torch.no_grad():
    x_pred = predict_x(u1_test, u2_test)
    y_pred = predict_y(u1_test, u2_test, u3_test)

mse_x = torch.mean((x_pred - x_true) ** 2).item()
mse_y = torch.mean((y_pred - y_true) ** 2).item()

print(f"\nMSE for x (on 1000 test samples): {mse_x:.4f}")
print(f"MSE for y (on 1000 test samples): {mse_y:.4f}")

Iter 1 | loss: 3.7107, s1: 4.6688, s2: 1.9163
Iter 1000 | loss: 1.7319, s1: 3.3809, s2: 3.2981
Iter 2000 | loss: 1.7061, s1: 3.3726, s2: 3.3331
Iter 3000 | loss: 1.6714, s1: 3.3694, s2: 3.3961
Iter 4000 | loss: 1.7137, s1: 3.4723, s2: 3.5172
Iter 5000 | loss: 1.7105, s1: 3.3921, s2: 3.3632
Iter 6000 | loss: 1.7072, s1: 3.4225, s2: 3.4306
Iter 7000 | loss: 1.6697, s1: 3.3555, s2: 3.3714
Iter 8000 | loss: 1.7270, s1: 3.3765, s2: 3.2990

True beta: [2, 1]
Estimated beta: [1.9821306467056274, 1.0577934980392456]
True intercept: [1]；Estimated intercept: [0.9996429681777954]
Beta estimation error: [0.01786935329437256, 0.057793498039245605]

MSE for x (on 1000 test samples): 0.0744
MSE for y (on 1000 test samples): 0.2997
